In [11]:
from astropy.table import Table

table = Table.read(r"C:\Users\DELL\Downloads\7406ca63-425e-11f1-8aa8-bc97e148b76b-O-result.vot\Gaiadatatonstars50k.vot", format="votable")
df = table.to_pandas()

In [12]:
df.head()
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   SOURCE_ID        50000 non-null  int64  
 1   teff_gspphot     50000 non-null  float32
 2   phot_g_mean_mag  50000 non-null  float32
 3   bp_rp            50000 non-null  float32
 4   parallax         50000 non-null  float64
dtypes: float32(3), float64(1), int64(1)
memory usage: 1.3 MB


,SOURCE_ID,teff_gspphot,phot_g_mean_mag,bp_rp,parallax
count,5.000000e+04,50000.000000,50000.000000,50000.000000,50000.000000
mean,3.294336e+15,4565.652832,16.886757,1.498621,1.281535
std,1.923893e+15,871.134827,1.787898,0.603532,1.475211
min,4.295807e+09,2737.746338,5.600941,-0.372690,-3.269365
25%,1.666947e+15,3765.183228,15.927801,1.015476,0.464259
50%,3.245483e+15,4577.219482,17.361628,1.301565,0.891100
75%,4.860807e+15,5211.375610,18.287718,1.928495,1.635712
max,7.906382e+15,19144.222656,18.999931,3.986099,116.267814


In [13]:
df = df[["teff_gspphot", "phot_g_mean_mag", "bp_rp", "parallax"]]
df.isnull().sum()
df.dropna()
df = df[
    (df["teff_gspphot"] > 2000) & (df["teff_gspphot"] < 40000) &   #extreme temperatures -> noise, negative parallax -> impossible, bp-rp -> bad measurements
    (df["parallax"] > 0) &
    (df["bp_rp"] > -1) & (df["bp_rp"] < 5)
]

In [14]:
import numpy as np

df["abs_mag"] = df["phot_g_mean_mag"] + 5 * np.log10(df["parallax"]/1000) + 5   #apparent brightness -> intrinsic brightness : M = m + 5 log10(d) − 5
                                                                                                                             # d ≈ 1 / parallax
df["log_temp"] = np.log10(df["teff_gspphot"])    #compressing the scale of temperature; HR diagrams are plotted in log scale

df["log_lum"] = -0.4 * df["abs_mag"]     #converting absolute magnitude to logarithm of luminosity #luminosity ~ 10^(-0.4 x magnitude)

df["temp_color_interaction"] = df["log_temp"] * df["bp_rp"]    #temperature in relation to color index <- interaction feature, helps capture non-linear structures

df = df[
    (df["abs_mag"] > -20) & (df["abs_mag"] < 30)          #mild filter
]

In [15]:
from sklearn.preprocessing import StandardScaler

features = ["teff_gspphot", "bp_rp", "abs_mag"]       #rescaling all the features

scaler = StandardScaler()
df[features] = scaler.fit_transform(df[features])

In [16]:
df.describe()
df.shape

(48407, 8)